# Analysis of Mirror-AI Results

This notebook provides a template for visualizing the metrics computed in `result.csv`.

In [ ]:
from pathlib import Path


CSV_PATH = Path("final_outputs/reports/result_trimmed.csv")

OUT_DIR = Path("figures")

OUT_DIR.mkdir(exist_ok=True)    

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



EXPERIMENTS = ["e1", "e2", "e3", "e5"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e2": "No mirror",
    "e3": "Wrong prior",
    "e5": "Distractors",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude\nSonnet 4.6",
    "gemini-2.5-flash": "Gemini\n2.5 Flash",
    "gemini-2.5-pro": "Gemini\n2.5 Pro",
    "gemma4_26b": "Gemma4\n26B",
    "gemma4_31b": "Gemma4\n31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral\n3 14B",
    "qwen3_6_plus": "Qwen\n3.6 Plus",
}


# ============================================================
# LOAD + CLEAN
# ============================================================

def load_results(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.astype(str).str.strip()

    if not {"experiment_id", "model_name"}.issubset(df.columns):
        raw = pd.read_csv(csv_path, header=None)
        df = raw.iloc[1:].copy()
        df.columns = raw.iloc[0].astype(str).str.strip()

    df = df.loc[:, ~df.columns.duplicated()]
    df = df.drop(
        columns=[
            c for c in df.columns
            if str(c).lower().startswith("unnamed") or str(c).lower() == "nan"
        ],
        errors="ignore",
    )

    df.columns = df.columns.astype(str).str.strip()

    for col in df.columns:
        if col not in ["experiment_id", "model_name"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["model_label"] = df["model_name"].map(MODEL_LABELS).fillna(df["model_name"])
    return df


df = load_results(CSV_PATH)
plot_df = df[df["experiment_id"].isin(EXPERIMENTS)].copy()


# ============================================================
# ORDER MODELS
# ============================================================

# Option A, recommended for Figure 1:
# order by baseline mirror performance e1.
model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

# Option B:
# order by average across e1/e2/e3/e5.
# model_order = (
#     plot_df.groupby("model_label")["TSA_mean"]
#     .mean()
#     .sort_values(ascending=False)
#     .index
#     .tolist()
# )


# ============================================================
# MAKE MATRIX
# ============================================================

heat = (
    plot_df
    .pivot(index="experiment_id", columns="model_label", values="TSA_mean")
    .reindex(EXPERIMENTS)
    .reindex(columns=model_order)
)

heat.index = [EXPERIMENT_LABELS[e] for e in heat.index]


# ============================================================
# PLOT
# ============================================================

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 9,
})

fig, ax = plt.subplots(figsize=(12, 2.9))

im = ax.imshow(
    heat.values,
    vmin=0,
    vmax=1,
    aspect="auto",
    cmap="YlGnBu",
)

# Axes labels
ax.set_xticks(np.arange(heat.shape[1]))
ax.set_xticklabels(heat.columns)

ax.set_yticks(np.arange(heat.shape[0]))
ax.set_yticklabels(heat.index)

ax.tick_params(axis="x", length=0, pad=6)
ax.tick_params(axis="y", length=0, pad=6)

# Remove spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Annotate cells as percentages
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        text_color = "white" if value >= 0.62 else "black"
        ax.text(
            j,
            i,
            f"{value:.0%}",
            ha="center",
            va="center",
            fontsize=8,
            color=text_color,
        )

# Thin separators between rows only, less boxy than full grid
for y in np.arange(0.5, heat.shape[0] - 0.5, 1):
    ax.axhline(y, color="white", linewidth=1.2)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("TSA")
cbar.outline.set_visible(False)

# Add chance reference on colorbar
chance = 1 / 3
cbar.ax.axhline(chance, color="black", linewidth=1.0, linestyle="--")
cbar.ax.text(
    1.8,
    chance,
    "chance",
    va="center",
    ha="left",
    fontsize=8,
    transform=cbar.ax.get_yaxis_transform(),
)

ax.set_title("Target selection accuracy across self-recognition tasks", pad=12)

fig.tight_layout()



plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path



EXPERIMENTS = ["e1", "e2", "e3", "e5"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e2": "No mirror",
    "e3": "Wrong prior",
    "e5": "Distractors",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
}


def load_results(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.astype(str).str.strip()

    if not {"experiment_id", "model_name"}.issubset(df.columns):
        raw = pd.read_csv(csv_path, header=None)
        df = raw.iloc[1:].copy()
        df.columns = raw.iloc[0].astype(str).str.strip()

    df = df.loc[:, ~df.columns.duplicated()]
    df = df.drop(
        columns=[
            c for c in df.columns
            if str(c).lower().startswith("unnamed") or str(c).lower() == "nan"
        ],
        errors="ignore",
    )

    for col in df.columns:
        if col not in ["experiment_id", "model_name"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["model_label"] = df["model_name"].map(MODEL_LABELS).fillna(df["model_name"])
    return df


df = load_results(CSV_PATH)

plot_df = df[df["experiment_id"].isin(EXPERIMENTS)].copy()

model_order = (
    plot_df.groupby("model_label")["TSA_mean"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

heat = (
    plot_df.pivot(index="model_label", columns="experiment_id", values="TSA_mean")
    .reindex(model_order)
    .reindex(columns=EXPERIMENTS)
)

fig, ax = plt.subplots(figsize=(7, 4.6))

im = ax.imshow(heat.values, aspect="auto", vmin=0, vmax=1)

ax.set_xticks(np.arange(len(EXPERIMENTS)))
ax.set_xticklabels([EXPERIMENT_LABELS[e] for e in EXPERIMENTS])

ax.set_yticks(np.arange(len(model_order)))
ax.set_yticklabels(model_order)

# Add values inside cells
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if value > 0.55 else "black",
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Target Selection Accuracy")

ax.set_title("Target selection accuracy across self-recognition tasks")

ax.set_xticks(np.arange(-0.5, len(EXPERIMENTS), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(model_order), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.2)
ax.tick_params(which="minor", bottom=False, left=False)

fig.tight_layout()

png_path = OUT_DIR / "plot1_tsa_heatmap_no_e7.png"
pdf_path = OUT_DIR / "plot1_tsa_heatmap_no_e7.pdf"

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")

plt.show()

print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# ASSUMPTION
# ============================================================
# This code assumes your results dataframe is already loaded as:
#
# df
#
# Example:
# df = pd.read_csv(...)
#
# No file path is used here.


# ============================================================
# CONFIG
# ============================================================

EXPERIMENTS = ["e1", "e2", "e3", "e5"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e2": "No mirror",
    "e3": "Wrong prior",
    "e5": "Distractors",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
}

RADAR_METRICS = {
    "TSA": {
        "source": "TSA_mean",
        "label": "Task\nsuccess",
        "higher_is_better": True,
    },
    "MCR": {
        "source": "MCR_mean",
        "label": "Mirror\nconsult",
        "higher_is_better": True,
    },
    "MGD": {
        "source": "MGD_mean",
        "label": "Mirror\ngaze",
        "higher_is_better": True,
    },
    "MTATO": {
        "source": "MTATO_mean",
        "label": "Mirror\nbefore act",
        "higher_is_better": True,
    },
    "CAAL": {
        "source": "CAAL_mean",
        "label": "Correct\nself-color",
        "higher_is_better": True,
    },
    "RCS": {
        "source": "RCS_mean",
        "label": "Identity\ncoherence",
        "higher_is_better": True,
    },
    "CONF": {
        "source": "CONFABULATION_mean",
        "label": "Epistemic\ncalibration",
        "higher_is_better": False,
    },
}


# ============================================================
# CLEAN EXISTING DF
# ============================================================

plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

required_columns = {"experiment_id", "model_name"}
missing_required = required_columns - set(plot_df.columns)
if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df["model_label"] = (
    plot_df["model_name"]
    .map(MODEL_LABELS)
    .fillna(plot_df["model_name"])
)

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

metric_columns = {spec["source"] for spec in RADAR_METRICS.values()}
missing_metrics = metric_columns - set(plot_df.columns)
if missing_metrics:
    raise ValueError(
        f"Missing metric columns: {missing_metrics}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()


# ============================================================
# NORMALIZATION
# ============================================================

norm_params = {}

def minmax_normalize_with_params(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or np.isclose(max_val, min_val):
        normed = pd.Series(0.5, index=series.index)
    else:
        normed = (series - min_val) / (max_val - min_val)

    return normed, min_val, max_val


for metric_name, spec in RADAR_METRICS.items():
    source = spec["source"]
    norm_col = f"{metric_name}_norm"

    normed, min_val, max_val = minmax_normalize_with_params(plot_df[source])

    if not spec["higher_is_better"]:
        normed = 1 - normed

    plot_df[norm_col] = normed

    norm_params[metric_name] = {
        "min": min_val,
        "max": max_val,
        "higher_is_better": spec["higher_is_better"],
    }


def normalize_reference_value(metric_name, raw_value):
    min_val = norm_params[metric_name]["min"]
    max_val = norm_params[metric_name]["max"]
    higher_is_better = norm_params[metric_name]["higher_is_better"]

    if np.isclose(max_val, min_val):
        normed = 0.5
    else:
        normed = (raw_value - min_val) / (max_val - min_val)

    if not higher_is_better:
        normed = 1 - normed

    return float(np.clip(normed, 0, 1))


# Random guessing is 1/3 because cube-selection tasks use 3 candidate colors.
chance_tsa_norm = normalize_reference_value("TSA", 1 / 3)


# ============================================================
# MODEL ORDER
# ============================================================

model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

# Fallback if e1 is missing for some reason
if len(model_order) == 0:
    model_order = (
        plot_df.groupby("model_label")["TSA_mean"]
        .mean()
        .sort_values(ascending=False)
        .index
        .tolist()
    )


# ============================================================
# STYLE
# ============================================================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "figure.titlesize": 14,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})


# ============================================================
# RADAR PLOT
# ============================================================

metric_keys = list(RADAR_METRICS.keys())
metric_labels = [RADAR_METRICS[m]["label"] for m in metric_keys]
norm_cols = [f"{m}_norm" for m in metric_keys]

n_metrics = len(metric_keys)

angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

tsa_angle = angles[0]

fig, axes = plt.subplots(
    1,
    len(EXPERIMENTS),
    subplot_kw={"projection": "polar"},
    figsize=(15.5, 4.8),
    constrained_layout=True,
)

if len(EXPERIMENTS) == 1:
    axes = [axes]

fig.set_constrained_layout_pads(
    w_pad=0.08,
    h_pad=0.08,
    hspace=0.05,
    wspace=0.14,
)

for ax, exp in zip(axes, EXPERIMENTS):
    sub_exp = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    # Put task success axis at the right and proceed clockwise.
    ax.set_theta_offset(0)
    ax.set_theta_direction(-1)

    # Random guess marker only on Task Success axis.
    ax.scatter(
        [tsa_angle],
        [chance_tsa_norm],
        s=45,
        color="black",
        marker="x",
        linewidths=1.7,
        zorder=20,
    )

    for model in model_order:
        values = sub_exp.loc[model, norm_cols].astype(float).values
        values = np.concatenate([values, [values[0]]])

        ax.plot(
            angles,
            values,
            linewidth=1.45,
            marker="o",
            markersize=3.0,
            alpha=0.74,
            label=model,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=14)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, fontsize=9)

    ax.set_ylim(0, 1)
    ax.set_yticks([0.5, 1.0])
    ax.set_yticklabels(["0.5", "1.0"], fontsize=8.5)
    ax.set_rlabel_position(18)

    ax.grid(alpha=0.25, linewidth=0.9)
    ax.spines["polar"].set_alpha(0.45)
    ax.spines["polar"].set_linewidth(0.9)


fig.suptitle(
    "Behavioral self-recognition profiles across task variants",
    y=1.03,
)

model_handles, model_labels = axes[0].get_legend_handles_labels()

guess_handle = Line2D(
    [0],
    [0],
    marker="x",
    color="black",
    linestyle="None",
    markersize=7,
    markeredgewidth=1.7,
    label="Random guess",
)

fig.legend(
    model_handles + [guess_handle],
    model_labels + ["Random guess"],
    loc="center left",
    bbox_to_anchor=(1.01, 0.52),
    frameon=False,
    ncol=1,
    title="Model",
    title_fontsize=10,
)


plt.show()


# ============================================================
# OPTIONAL EXPORT
# ============================================================

# fig.savefig(
#     "radar_profiles_clean_large.pdf",
#     bbox_inches="tight",
#     pad_inches=0.05,
# )
#
# fig.savefig(
#     "radar_profiles_clean_large.png",
#     dpi=500,
#     bbox_inches="tight",
#     pad_inches=0.05,
# )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# CONFIG
# ============================================================
EXPERIMENTS = ["e1", "e2", "e3", "e5"]

# Paper-facing experiment names (not internal text labels)
# Here e5 is the 4th plotted condition, so it is shown as E4.
EXPERIMENT_LABELS = {
    "e1": "E1",
    "e2": "E2",
    "e3": "E3",
    "e5": "E4",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral-3_14b": "Ministral 3 14B",
    "qwen_qwen3.6-plus": "Qwen 3.6 Plus",
}

# Short metric labels only
# Higher is better on all plotted axes.
# TTD is reversed into a "faster is better" score.
# CR is reversed into a "lower confabulation is better" score.
RADAR_METRICS = {
    "TSA": {"source": "TSA_mean", "label": "TSA", "transform": "identity"},
    "TTD": {"source": "TTD_mean", "label": "TTD", "transform": "reverse_minmax"},
    "MCR": {"source": "MCR_mean", "label": "MCR", "transform": "identity"},
    "CAAL": {"source": "CAAL_mean", "label": "CAAL", "transform": "identity"},
    "CR": {"source": "CONFABULATION_mean", "label": "CR", "transform": "reverse_identity"},
}

# ============================================================
# DATA PREP
# ============================================================
plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = plot_df["model_name"].map(MODEL_LABELS).fillna(plot_df["model_name"])

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()

# ============================================================
# TRANSFORMS
# ============================================================
ttd_min = float(plot_df["TTD_mean"].min())
ttd_max = float(plot_df["TTD_mean"].max())

def reverse_minmax(series, min_val, max_val):
    if np.isclose(max_val, min_val):
        return pd.Series(0.5, index=series.index)
    return 1 - (series - min_val) / (max_val - min_val)

for metric_name, spec in RADAR_METRICS.items():
    source = spec["source"]
    out_col = f"{metric_name}_plot"

    if spec["transform"] == "identity":
        plot_df[out_col] = plot_df[source].clip(0, 1)
    elif spec["transform"] == "reverse_identity":
        plot_df[out_col] = (1 - plot_df[source]).clip(0, 1)
    elif spec["transform"] == "reverse_minmax":
        plot_df[out_col] = reverse_minmax(plot_df[source], ttd_min, ttd_max).clip(0, 1)
    else:
        raise ValueError(f"Unknown transform for {metric_name}: {spec['transform']}")

# Chance-level TSA for 3 cubes
chance_tsa_value = 1 / 3

# ============================================================
# ORDERING
# ============================================================
model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

if len(model_order) == 0:
    model_order = (
        plot_df.groupby("model_label")["TSA_mean"]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )

# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "figure.titlesize": 18,
    "legend.fontsize": 11,
    "font.family": "serif",
})

COLORS = [
    "#0072B2",  # blue
    "#D55E00",  # orange
    "#009E73",  # green
    "#CC79A7",  # magenta
    "#E69F00",  # mustard
    "#56B4E9",  # light blue
    "#6F4E7C",  # muted purple
    "#4D4D4D",  # dark gray
]

MARKERS = ["o", "s", "^", "D", "P", "X", "v", "h"]

metric_keys = list(RADAR_METRICS.keys())
metric_labels = [RADAR_METRICS[m]["label"] for m in metric_keys]
plot_cols = [f"{m}_plot" for m in metric_keys]
n_metrics = len(metric_keys)

# Start from top, clockwise
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False)
angles_closed = np.concatenate([angles, [angles[0]]])

# TSA is the first spoke
tsa_angle = angles[0]

# ============================================================
# FIGURE
# ============================================================
fig, axes = plt.subplots(
    1,
    len(EXPERIMENTS),
    subplot_kw={"projection": "polar"},
    figsize=(18, 6.8),
    constrained_layout=True,
)

if len(EXPERIMENTS) == 1:
    axes = [axes]

fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.03, wspace=0.08)

for ax, exp in zip(axes, EXPERIMENTS):
    sub_exp = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_facecolor("#FCFCFC")

    # Chance-level TSA marker
    ax.scatter(
        [tsa_angle],
        [chance_tsa_value],
        s=80,
        color="#C00000",
        marker="X",
        linewidths=1.4,
        edgecolor="black",
        zorder=60,
    )

    # Model lines
    for idx, model in enumerate(model_order):
        values = sub_exp.loc[model, plot_cols].astype(float).values
        values = np.nan_to_num(values, nan=0.0)
        values_closed = np.concatenate([values, [values[0]]])

        ax.plot(
            angles_closed,
            values_closed,
            color=COLORS[idx],
            linewidth=2.2,
            linestyle="-",
            marker=MARKERS[idx],
            markersize=8.5,
            markerfacecolor=COLORS[idx],
            markeredgecolor="white",
            markeredgewidth=1.0,
            alpha=0.95,
            label=model,
            zorder=10 + idx,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=20, fontweight="bold")

    # Spoke labels: short only, larger for readability
    ax.set_xticks(angles)
    ax.set_xticklabels(metric_labels, fontsize=12.5, fontweight="bold")
    ax.tick_params(axis="x", pad=14)

    # Radius / grid
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(["", "0.5", "", "1.0"], fontsize=9.5, color="#666666")
    ax.set_rlabel_position(20)

    ax.grid(alpha=0.35, linewidth=1.0, color="#9A9A9A")
    ax.spines["polar"].set_alpha(0.7)
    ax.spines["polar"].set_linewidth(1.1)
    ax.spines["polar"].set_color("#888888")

fig.suptitle(
    "Behavioral self-recognition profiles across task variants",
    y=1.05,
    fontweight="bold",
)

# Small note for the transformed axes
fig.text(
    0.5,
    -0.015,
    "Higher is better on all axes. TTD is reversed (faster decisions score higher); CR is reversed (lower confabulation scores higher).",
    ha="center",
    va="top",
    fontsize=10.5,
    color="#444444",
)

# ============================================================
# LEGEND
# ============================================================
model_handles = [
    Line2D(
        [0], [0],
        color=COLORS[idx],
        linewidth=2.2,
        linestyle="-",
        marker=MARKERS[idx],
        markersize=8,
        markerfacecolor=COLORS[idx],
        markeredgecolor="white",
        markeredgewidth=1.0,
        label=model,
    )
    for idx, model in enumerate(model_order)
]

chance_handle = Line2D(
    [0], [0],
    marker="X",
    color="w",
    markerfacecolor="#C00000",
    markeredgecolor="black",
    markersize=9,
    linewidth=0,
    label="Chance-level TSA",
)

legend = fig.legend(
    model_handles + [chance_handle],
    model_order + ["Chance-level TSA"],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.20),
    ncol=3,
    frameon=True,
    fancybox=True,
    framealpha=0.96,
    edgecolor="#D0D0D0",
    facecolor="white",
    title="Models",
    title_fontproperties={"weight": "bold", "size": 12},
    columnspacing=1.6,
    handlelength=2.0,
    handletextpad=0.6,
    borderpad=0.8,
)

# ============================================================
# EXPORT
# ============================================================
fig.savefig("radar_profiles_paper_clean.pdf", format="pdf", bbox_inches="tight", pad_inches=0.08)
fig.savefig("radar_profiles_paper_clean.png", dpi=500, bbox_inches="tight", pad_inches=0.08)

plt.show()


In [ ]:
# ============================================================
# Extended appendix table in the same LaTeX style as tab:all-results
# ============================================================

import math
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd

RESULTS_CSV = CSV_PATH
OUT_TEX = Path("paper_assets/appendix_extended_metrics_table.tex")
OUT_TEX.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RESULTS_CSV)

# Paper-facing experiments only
EXP_ORDER = ["e1", "e2", "e3", "e5", "e7"]

EXP_TITLES = OrderedDict([
    ("e1", "E1: Active mirror-based self-identification"),
    ("e2", "E2: Self-identification without mirror evidence"),
    ("e3", "E3: Self-identification under conflicting linguistic information"),
    ("e5", "E4: Self--other disambiguation under visual ambiguity"),
    ("e7", "E5: Robust self-identification under occlusion"),  # change here if you updated the title in the paper
])

MODEL_ORDER = [
    "ministral-3_14b",
    "gemma4_26b",
    "gemma4_31b",
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "qwen_qwen3.6-plus",
    "gpt-5.1",
    "anthropic_claude-sonnet-4.6",
]

MODEL_LABELS = {
    "ministral-3_14b": "Ministral 3 14B",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "qwen_qwen3.6-plus": "Qwen 3.6 Plus",
    "gpt-5.1": "GPT 5.1",
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
}

# ------------------------------------------------------------------
# Columns
# rank_dir:
#   "max" -> bold best / underline second-best, larger is better
#   "min" -> bold best / underline second-best, smaller is better
#   None  -> no ranking decoration
# ------------------------------------------------------------------
METRICS = [
    {
        "mean": "TSA_mean",
        "se": "TSA_se",
        "header": r"\textbf{TSA} $\uparrow$",
        "rank_dir": "max",
        "display_name": "TSA",
    },
    {
        "mean": "TSA-COMPLETE_mean",
        "se": "TSA-COMPLETE_se",
        "header": r"\textbf{TSA-C} $\uparrow$",
        "rank_dir": "max",
        "display_name": "TSA-C",
    },
    {
        "mean": "MCR_mean",
        "se": "MCR_se",
        "header": r"\textbf{MCR} $\uparrow$",
        "rank_dir": "max",
        "display_name": "MCR",
    },
    {
        "mean": "MGD_mean",
        "se": "MGD_se",
        "header": r"\textbf{MGD} $\uparrow$",
        "rank_dir": "max",
        "display_name": "MGD",
    },
    {
        "mean": "CAAL_mean",
        "se": "CAAL_se",
        "header": r"\textbf{CAAL} $\uparrow$",
        "rank_dir": "max",
        "display_name": "CAAL",
    },
    {
        "mean": "SELF_CORRECTION_mean",
        "se": "SELF_CORRECTION_se",
        "header": r"\textbf{SC} $\uparrow$",
        "rank_dir": "max",
        "display_name": "SC",
    },
    {
        "mean": "AR_SC_mean",
        "se": "AR_SC_se",
        "header": r"\textbf{AR$_{\mathrm{SC}}$} $\uparrow$",
        "rank_dir": "max",
        "display_name": "AR_SC",
    },
    {
        "mean": "RCS_mean",
        "se": "RCS_se",
        "header": r"\textbf{RCS} $\uparrow$",
        "rank_dir": "max",
        "display_name": "RCS",
    },
    {
        "mean": "EUL_mean",
        "se": "EUL_se",
        "header": r"\textbf{EUL} $\downarrow$",
        "rank_dir": "min",
        "display_name": "EUL",
    },
    {
        "mean": "SRR_mean",
        "se": "SRR_se",
        "header": r"\textbf{SRR} $\uparrow$",
        "rank_dir": "max",
        "display_name": "SRR",
    },
]


# Compute AR_SC se from Bernoulli rate over total runs
df["AR_SC_mean"] = df["SELF_CORRECTION_applicable_rate"]
df["AR_SC_se"] = np.where(
    df["total_runs"] > 1,
    np.sqrt(df["AR_SC_mean"] * (1 - df["AR_SC_mean"]) / df["total_runs"]),
    0.0,
)

df = df[df["experiment_id"].isin(EXP_ORDER)].copy()

def distinct_best_second(values, direction="max", tol=1e-12):
    vals = [v for v in values if pd.notna(v)]
    if not vals:
        return None, None
    uniq = sorted(set(round(float(v), 12) for v in vals), reverse=(direction == "max"))
    best = uniq[0]
    second = uniq[1] if len(uniq) > 1 else None
    return best, second

def value_is_equal(a, b, tol=1e-12):
    if a is None or b is None or pd.isna(a):
        return False
    return abs(float(a) - float(b)) <= tol

def format_metric(row, metric, best=None, second=None):
    mean_col = metric["mean"]
    se_col = metric["se"]
    val = row[mean_col]
    se = row[se_col]

    # SC is undefined when applicability rate is zero
    if mean_col == "SELF_CORRECTION_mean" and row["AR_SC_mean"] == 0:
        return r"--"

    if pd.isna(val):
        return r"--"

    cell = rf"${val:.2f} \pm {se:.2f}$"

    if metric["rank_dir"] is not None:
        if value_is_equal(val, best):
            cell = r"$\mathbf{" + f"{val:.2f} \\pm {se:.2f}" + r"}$"
        elif second is not None and value_is_equal(val, second):
            cell = r"$\underline{" + f"{val:.2f} \\pm {se:.2f}" + r"}$"

    return cell

# Precompute ranking thresholds per experiment per metric
rank_lookup = {}
for exp in EXP_ORDER:
    exp_df = (
        df[df["experiment_id"] == exp]
        .set_index("model_name")
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    rank_lookup[exp] = {}
    for metric in METRICS:
        if metric["rank_dir"] is None:
            rank_lookup[exp][metric["mean"]] = (None, None)
            continue

        vals = []
        for _, row in exp_df.iterrows():
            # skip undefined SC
            if metric["mean"] == "SELF_CORRECTION_mean" and row["AR_SC_mean"] == 0:
                continue
            vals.append(row[metric["mean"]])

        best, second = distinct_best_second(vals, metric["rank_dir"])
        rank_lookup[exp][metric["mean"]] = (best, second)

# Build LaTeX
ncols = 1 + len(METRICS)
tabspec = "l" + "c" * len(METRICS)

lines = []
lines.append(r"\begin{table*}[t]")
lines.append(r"\centering")
lines.append(r"%\scriptsize")
lines.append(r"\resizebox{\textwidth}{!}{%")
lines.append(r"\begin{tabular}{" + tabspec + "}")
lines.append(r"\toprule")
lines.append(
    r"\textbf{Model} & " + " & ".join(m["header"] for m in METRICS) + r" \\"
)
lines.append(r"\midrule")
lines.append("")

for exp in EXP_ORDER:
    exp_df = (
        df[df["experiment_id"] == exp]
        .set_index("model_name")
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    lines.append(r"\rowcolor{gray!15}")
    lines.append(
        rf"\multicolumn{{{ncols}}}{{c}}{{\textbf{{{EXP_TITLES[exp]}}}}} \\"
    )
    lines.append(r"\midrule")

    for _, row in exp_df.iterrows():
        model_name = MODEL_LABELS[row["model_name"]]
        formatted = []
        for metric in METRICS:
            best, second = rank_lookup[exp][metric["mean"]]
            formatted.append(format_metric(row, metric, best, second))

        lines.append(
            rf"\texttt{{{model_name}}} & " + " & ".join(formatted) + r" \\"
        )

    if exp != EXP_ORDER[-1]:
        lines.append("")
        lines.append(r"\midrule")

lines.append("")
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}%")
lines.append(r"}")
lines.append(
    r"\caption{\textbf{Extended appendix metrics across all defined settings}. "
    r"Values are mean $\pm$ standard error. Best results are in bold and "
    r"second-best results are underlined for every reported metric. "
    r"SC is reported as -- when self-correction is not applicable.}"
)

lines.append(r"\label{tab:appendix-extended-metrics}")
lines.append(r"\end{table*}")

latex = "\n".join(lines)
OUT_TEX.write_text(latex)

print(f"Wrote {OUT_TEX.resolve()}\n")
print(latex[:5000] + ("\n...\n" if len(latex) > 5000 else ""))


In [ ]:
# ============================================================
# Extended appendix table with the same LaTeX layout/style
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from collections import OrderedDict

RESULTS_CSV = CSV_PATH
OUT_TEX = Path("paper_assets/appendix_extended_metrics_table.tex")
OUT_TEX.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RESULTS_CSV)

EXP_ORDER = ["e1", "e2", "e3", "e5", "e7"]

EXP_TITLES = OrderedDict([
    ("e1", "E1: Active mirror-based self-identification"),
    ("e2", "E2: Self-identification without mirror evidence"),
    ("e3", "E3: Self-identification under conflicting linguistic information"),
    ("e5", "E4: Self--other disambiguation under visual ambiguity"),
    ("e7", "E5: Robust self-identification under occlusion"),  # change if you renamed E5 in the paper
])

MODEL_ORDER = [
    "ministral-3_14b",
    "gemma4_26b",
    "gemma4_31b",
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "qwen_qwen3.6-plus",
    "gpt-5.1",
    "anthropic_claude-sonnet-4.6",
]

MODEL_LABELS = {
    "ministral-3_14b": "Ministral 3 14B",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "qwen_qwen3.6-plus": "Qwen 3.6 Plus",
    "gpt-5.1": "GPT 5.1",
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
}

# All metrics ranked within each experiment
# Convention:
# - higher is better for TSA, TSA-C, MCR, MGD, CAAL, SC, AR_SC
# - SC is shown as -- when AR_SC = 0, and excluded from ranking in that case
METRICS = [
    {
        "mean": "TSA_mean",
        "se": "TSA_se",
        "header": r"\textbf{TSA} $\uparrow$",
        "rank_dir": "max",
    },
    {
        "mean": "TSA-COMPLETE_mean",
        "se": "TSA-COMPLETE_se",
        "header": r"\textbf{TSA-C} $\uparrow$",
        "rank_dir": "max",
    },
    {
        "mean": "MCR_mean",
        "se": "MCR_se",
        "header": r"\textbf{MCR} $\uparrow$",
        "rank_dir": "max",
    },
    {
        "mean": "MGD_mean",
        "se": "MGD_se",
        "header": r"\textbf{MGD} $\uparrow$",
        "rank_dir": "max",
    },
    {
        "mean": "CAAL_mean",
        "se": "CAAL_se",
        "header": r"\textbf{CAAL} $\uparrow$",
        "rank_dir": "max",
    },
    {
        "mean": "SELF_CORRECTION_mean",
        "se": "SELF_CORRECTION_se",
        "header": r"\textbf{SC} $\uparrow$",
        "rank_dir": "max",
    },
    {
        "mean": "AR_SC_mean",
        "se": "AR_SC_se",
        "header": r"\textbf{AR$_{\mathrm{SC}}$} $\uparrow$",
        "rank_dir": "max",
    },
]

# Derived mean/se for AR_SC from applicability rate and total_runs
df["AR_SC_mean"] = df["SELF_CORRECTION_applicable_rate"]
df["AR_SC_se"] = np.where(
    df["total_runs"] > 1,
    np.sqrt(df["AR_SC_mean"] * (1 - df["AR_SC_mean"]) / df["total_runs"]),
    0.0,
)

df = df[df["experiment_id"].isin(EXP_ORDER)].copy()

def distinct_best_second(values, direction="max", tol=1e-12):
    vals = [v for v in values if pd.notna(v)]
    if not vals:
        return None, None
    uniq = sorted(set(round(float(v), 12) for v in vals), reverse=(direction == "max"))
    best = uniq[0]
    second = uniq[1] if len(uniq) > 1 else None
    return best, second

def value_is_equal(a, b, tol=1e-12):
    if a is None or b is None or pd.isna(a):
        return False
    return abs(float(a) - float(b)) <= tol

def format_metric(row, metric, best=None, second=None):
    mean_col = metric["mean"]
    se_col = metric["se"]
    val = row[mean_col]
    se = row[se_col]

    # SC undefined when no applicable self-correction episodes exist
    if mean_col == "SELF_CORRECTION_mean" and row["AR_SC_mean"] == 0:
        return r"--"

    if pd.isna(val):
        return r"--"

    if value_is_equal(val, best):
        return r"$\mathbf{" + f"{val:.2f} \\pm {se:.2f}" + r"}$"
    if second is not None and value_is_equal(val, second):
        return r"$\underline{" + f"{val:.2f} \\pm {se:.2f}" + r"}$"
    return rf"${val:.2f} \pm {se:.2f}$"

# Ranking thresholds by experiment and metric
rank_lookup = {}
for exp in EXP_ORDER:
    exp_df = (
        df[df["experiment_id"] == exp]
        .set_index("model_name")
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    rank_lookup[exp] = {}
    for metric in METRICS:
        vals = []
        for _, row in exp_df.iterrows():
            if metric["mean"] == "SELF_CORRECTION_mean" and row["AR_SC_mean"] == 0:
                continue
            vals.append(row[metric["mean"]])

        best, second = distinct_best_second(vals, metric["rank_dir"])
        rank_lookup[exp][metric["mean"]] = (best, second)

# Build LaTeX
ncols = 1 + len(METRICS)
tabspec = "l" + "c" * len(METRICS)

lines = []
lines.append(r"\begin{table*}[t]")
lines.append(r"\centering")
lines.append(r"%\scriptsize")
lines.append(r"\resizebox{\textwidth}{!}{%")
lines.append(r"\begin{tabular}{" + tabspec + "}")
lines.append(r"\toprule")
lines.append(
    r"\textbf{Model} & " + " & ".join(m["header"] for m in METRICS) + r" \\"
)
lines.append(r"\midrule")
lines.append("")

for exp in EXP_ORDER:
    exp_df = (
        df[df["experiment_id"] == exp]
        .set_index("model_name")
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    lines.append(r"\rowcolor{gray!15}")
    lines.append(
        rf"\multicolumn{{{ncols}}}{{c}}{{\textbf{{{EXP_TITLES[exp]}}}}} \\"
    )
    lines.append(r"\midrule")

    for _, row in exp_df.iterrows():
        model_name = MODEL_LABELS[row["model_name"]]
        formatted = []
        for metric in METRICS:
            best, second = rank_lookup[exp][metric["mean"]]
            formatted.append(format_metric(row, metric, best, second))

        lines.append(
            rf"\texttt{{{model_name}}} & " + " & ".join(formatted) + r" \\"
        )

    if exp != EXP_ORDER[-1]:
        lines.append("")
        lines.append(r"\midrule")

lines.append("")
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}%")
lines.append(r"}")
lines.append(
    r"\caption{\textbf{Extended appendix metrics across all defined settings}. "
    r"Values are reported as mean $\pm$ standard error. Best results are shown "
    r"in bold and second-best results are underlined within each experiment and "
    r"metric. TSA-C denotes completion-conditioned task success. MGD denotes "
    r"mirror gaze duration. SC denotes the self-correction rate conditioned on "
    r"episodes where a wrong pre-mirror self-attribution is made and later "
    r"mirror evidence becomes available; it is reported as -- when this "
    r"condition is never met. AR$_{\mathrm{SC}}$ denotes the corresponding "
    r"applicability rate.}"
)
lines.append(r"\label{tab:appendix-extended-metrics}")
lines.append(r"\end{table*}")

latex = "\n".join(lines)
OUT_TEX.write_text(latex)

print(f"Wrote {OUT_TEX.resolve()}\n")
print(latex[:5000] + ("\n...\n" if len(latex) > 5000 else ""))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# CONFIG
# ============================================================
EXPERIMENTS = ["e1", "e2", "e3", "e5", "e7"]

# Paper-facing experiment IDs
EXPERIMENT_LABELS = {
    "e1": "E1",
    "e2": "E2",
    "e3": "E3",
    "e5": "E4",
    "e7": "E5",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma 4 26B",
    "gemma4_31b": "Gemma 3 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral-3_14b": "Ministral-3 14B",
    "qwen_qwen3.6-plus": "Qwen 3.6+",
}

# Higher is better on all plotted axes
RADAR_METRICS = {
    "TSA":   {"source": "TSA_mean",            "label": "TSA",   "transform": "identity"},
    "TTD":   {"source": "TTD_mean",            "label": "TTD",   "transform": "reverse_minmax"},
    "MCR":   {"source": "MCR_mean",            "label": "MCR",   "transform": "identity"},
    "MTATO": {"source": "MTATO_mean",          "label": "MTATO", "transform": "identity"},
    "CAAL":  {"source": "CAAL_mean",           "label": "CAAL",  "transform": "identity"},
    "CR":    {"source": "CONFABULATION_mean",  "label": "CR",    "transform": "reverse_identity"},
}

# Chance baseline:
# - cube tasks (E1-E4 / internal e1,e2,e3,e5): 1/3
# - exploration self-attribution task (E5 / internal e7): 1/15
CHANCE_TSA_BY_EXP = {
    "e1": 1 / 3,
    "e2": 1 / 3,
    "e3": 1 / 3,
    "e5": 1 / 3,
}

# Top models get slightly more visual weight, others remain fully visible
N_TOP_MODELS = 3

# ============================================================
# DATA PREP
# ============================================================
plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = plot_df["model_name"].map(MODEL_LABELS).fillna(plot_df["model_name"])

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()

# ============================================================
# TRANSFORMS
# ============================================================
ttd_min = float(plot_df["TTD_mean"].min())
ttd_max = float(plot_df["TTD_mean"].max())

def reverse_minmax(series, min_val, max_val):
    if np.isclose(max_val, min_val):
        return pd.Series(0.5, index=series.index)
    return 1 - (series - min_val) / (max_val - min_val)

for metric_name, spec in RADAR_METRICS.items():
    source = spec["source"]
    out_col = f"{metric_name}_plot"

    if spec["transform"] == "identity":
        plot_df[out_col] = plot_df[source].clip(0, 1)
    elif spec["transform"] == "reverse_identity":
        plot_df[out_col] = (1 - plot_df[source]).clip(0, 1)
    elif spec["transform"] == "reverse_minmax":
        plot_df[out_col] = reverse_minmax(plot_df[source], ttd_min, ttd_max).clip(0, 1)
    else:
        raise ValueError(f"Unknown transform for {metric_name}: {spec['transform']}")

# ============================================================
# ORDERING
# ============================================================
model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

if len(model_order) == 0:
    model_order = (
        plot_df.groupby("model_label")["TSA_mean"]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )

top_models = set(model_order[:N_TOP_MODELS])

# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.size": 11.5,
    "axes.titlesize": 15,
    "figure.titlesize": 18,
    "legend.fontsize": 11,
    "font.family": "serif",
})

COLORS = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # green
    "#CC79A7",  # magenta
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#6A3D9A",  # purple
    "#4D4D4D",  # dark gray
]

MARKERS = ["o", "s", "^", "D", "P", "X", "v", "h"]

metric_keys = list(RADAR_METRICS.keys())
metric_labels = [RADAR_METRICS[m]["label"] for m in metric_keys]
plot_cols = [f"{m}_plot" for m in metric_keys]
n_metrics = len(metric_keys)

# Start at top, go clockwise
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False)
angles_closed = np.concatenate([angles, [angles[0]]])

# TSA is first spoke
tsa_angle = angles[0]

# ============================================================
# FIGURE
# ============================================================
fig, axes = plt.subplots(
    1,
    len(EXPERIMENTS),
    subplot_kw={"projection": "polar"},
    figsize=(21.5, 5.8),   # was (18.5, 4.9)
    constrained_layout=False,
)

if len(EXPERIMENTS) == 1:
    axes = [axes]

fig.subplots_adjust(
    left=0.02,
    right=0.995,
    top=0.92,      # more room for the plots
    bottom=0.15,
    wspace=0.18,   # less wasted gap between panels
)

for ax, exp in zip(axes, EXPERIMENTS):
    sub_exp = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_facecolor("#FCFCFC")

    if exp != "e7":
        chance_tsa_value = CHANCE_TSA_BY_EXP[exp]

        ax.scatter(
            [tsa_angle],
            [chance_tsa_value],
            s=68,
            color="#C00000",
            marker="X",
            linewidths=1.2,
            edgecolor="black",
            zorder=80,
        )

    for idx, model in enumerate(model_order):
        values = sub_exp.loc[model, plot_cols].astype(float).values
        values = np.nan_to_num(values, nan=0.0)
        values_closed = np.concatenate([values, [values[0]]])

        is_top = model in top_models

        ax.plot(
            angles_closed,
            values_closed,
            color=COLORS[idx],
            linewidth=2.3 if is_top else 1.25,   # thinner non-top lines
            linestyle="-",
            marker=MARKERS[idx],
            markersize=7.0 if is_top else 5.8,   # slightly smaller non-top markers
            markerfacecolor=COLORS[idx],
            markeredgecolor="white",
            markeredgewidth=0.85,
            alpha=0.94 if is_top else 0.55,      # much less spaghetti
            label=model,
            zorder=20 + idx if is_top else 10 + idx,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=12, fontweight="bold")

    ax.set_xticks(angles)
    ax.set_xticklabels(metric_labels, fontsize=10.8, fontweight="bold")
    ax.tick_params(axis="x", pad=7)   # bring labels closer, frees inner space

    if exp == "e2":
        xticklabels = ax.get_xticklabels()
        for key in ["MCR", "MTATO"]:
            idx = metric_keys.index(key)
            xticklabels[idx].set_color("#999999")

    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(["", "0.5", "", "1.0"], fontsize=8.0, color="#666666")
    ax.set_rlabel_position(18)

    ax.grid(alpha=0.28, linewidth=0.9, color="#9A9A9A")
    ax.spines["polar"].set_alpha(0.68)
    ax.spines["polar"].set_linewidth(1.0)
    ax.spines["polar"].set_color("#888888")

# IMPORTANT: remove the suptitle entirely if the caption already says it
# fig.suptitle(...)

# ============================================================
# LEGEND
# ============================================================
legend_handles = []
legend_labels = []

for idx, model in enumerate(model_order):
    is_top = model in top_models
    legend_handles.append(
        Line2D(
            [0], [0],
            color=COLORS[idx],
            linewidth=2.3 if is_top else 1.25,
            linestyle="-",
            marker=MARKERS[idx],
            markersize=6.7 if is_top else 5.8,
            markerfacecolor=COLORS[idx],
            markeredgecolor="white",
            markeredgewidth=0.85,
            alpha=0.94 if is_top else 0.55,
            label=model,
        )
    )
    legend_labels.append(model)

chance_handle = Line2D(
    [0], [0],
    marker="X",
    color="w",
    markerfacecolor="#C00000",
    markeredgecolor="black",
    markersize=8.0,
    linewidth=0,
    label="Chance TSA",
)

fig.legend(
    legend_handles + [chance_handle],
    legend_labels + ["Chance TSA"],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=9,
    frameon=False,
    columnspacing=1.1,
    handlelength=1.7,
    handletextpad=0.35,
    borderpad=0.05,
    labelspacing=0.55,
    fontsize=15.0,
)


# ============================================================
# EXPORT
# ============================================================
fig.savefig("radar_profiles_final.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
fig.savefig("radar_profiles_compact.png", dpi=500, bbox_inches="tight", pad_inches=0.05)

plt.show()


In [ ]:
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# CONFIG
# ============================================================
RESULTS_CSV = CSV_PATH
OUTDIR = Path("paper_assets/appendix_plots")
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ORDER = ["e1", "e2", "e3", "e5", "e7"]

EXPERIMENT_LABELS = {
    "e1": "E1",
    "e2": "E2",
    "e3": "E3",
    "e5": "E4",
    "e7": "E5",
}

MODEL_STYLES = OrderedDict([
    ("anthropic_claude-sonnet-4.6", {"label": "Claude Sonnet 4.6", "color": "#0072B2", "marker": "o"}),
    ("qwen_qwen3.6-plus",           {"label": "Qwen 3.6+",         "color": "#D55E00", "marker": "s"}),
    ("gemma4_26b",                  {"label": "Gemma 4 26B",       "color": "#009E73", "marker": "^"}),
    ("gemma4_31b",                  {"label": "Gemma 4 31B",       "color": "#CC79A7", "marker": "D"}),
    ("gpt-5.1",                     {"label": "GPT-5.1",           "color": "#E69F00", "marker": "P"}),
    ("gemini-2.5-flash",            {"label": "Gemini 2.5 Flash",  "color": "#56B4E9", "marker": "X"}),
    ("gemini-2.5-pro",              {"label": "Gemini 2.5 Pro",    "color": "#6A3D9A", "marker": "v"}),
    ("ministral-3_14b",             {"label": "Ministral-3 14B",   "color": "#6B6B6B", "marker": "h"}),
])

MODEL_ORDER = list(MODEL_STYLES.keys())


# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
})


# ============================================================
# LOAD
# ============================================================
df = pd.read_csv(RESULTS_CSV)
df = df[df["experiment_id"].isin(EXPERIMENT_ORDER)].copy()
df["paper_exp"] = df["experiment_id"].map(EXPERIMENT_LABELS)


# ============================================================
# HELPERS
# ============================================================
def style_axis(ax):
    ax.set_facecolor("#FAFAF8")

    ax.grid(
        True,
        color="#D9D9D6",
        linewidth=0.65,
        alpha=0.55,
    )
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color("#777777")
    ax.spines["bottom"].set_color("#777777")
    ax.spines["left"].set_linewidth(0.75)
    ax.spines["bottom"].set_linewidth(0.75)

    ax.tick_params(
        axis="both",
        length=3,
        width=0.75,
        colors="#333333",
        pad=3,
    )


def model_legend_handles():
    return [
        Line2D(
            [0],
            [0],
            color="none",
            marker=spec["marker"],
            linestyle="None",
            markersize=6.8,
            markerfacecolor=spec["color"],
            markeredgecolor="white",
            markeredgewidth=0.85,
            label=spec["label"],
        )
        for spec in MODEL_STYLES.values()
    ]


# ============================================================
# TTD PERCENTAGE BINNING
# ============================================================
ttd_valid = df["TTD_mean"].dropna()


def ttd_percentile(ttd):
    if pd.isna(ttd) or len(ttd_valid) == 0:
        return np.nan

    return 100.0 * (ttd_valid <= ttd).mean()


def ttd_size_bin(ttd):
    pct = ttd_percentile(ttd)

    if pd.isna(pct):
        return 70, "unknown"

    if pct < 20:
        return 58, "Fastest 20%"

    if pct < 60:
        return 118, "Middle 40%"

    return 205, "Slowest 40%"


def size_legend_handles():
    return [
        plt.scatter([], [], s=58,  color="#9A9A9A", alpha=0.42, edgecolors="none", label="Fastest 20%"),
        plt.scatter([], [], s=118, color="#9A9A9A", alpha=0.42, edgecolors="none", label="Middle 40%"),
        plt.scatter([], [], s=205, color="#9A9A9A", alpha=0.42, edgecolors="none", label="Slowest 40%"),
    ]


# ============================================================
# FIGURE
# ============================================================
fig, axes = plt.subplots(
    1,
    len(EXPERIMENT_ORDER),
    figsize=(16.8, 4.25),
    sharex=True,
    sharey=True,
)

fig.patch.set_facecolor("white")


# ============================================================
# PANELS
# ============================================================
for ax, exp_id in zip(axes, EXPERIMENT_ORDER):
    sub = (
        df[df["experiment_id"] == exp_id]
        .set_index("model_name")
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    style_axis(ax)

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle=(0, (4, 3)),
        color="#9B9B9B",
        linewidth=0.9,
        alpha=0.65,
        zorder=1,
    )

    for _, row in sub.iterrows():
        model_id = row["model_name"]

        if model_id not in MODEL_STYLES:
            continue

        spec = MODEL_STYLES[model_id]
        size, _ = ttd_size_bin(row["TTD_mean"])

        ax.scatter(
            row["CAAL_mean"],
            row["TSA_mean"],
            s=size,
            color=spec["color"],
            marker=spec["marker"],
            alpha=0.86,
            edgecolors="white",
            linewidths=0.95,
            zorder=3,
        )

    ax.set_title(
        EXPERIMENT_LABELS[exp_id],
        fontweight="semibold",
        color="#222222",
        pad=8,
    )

    ax.set_xlim(-0.035, 1.035)
    ax.set_ylim(-0.035, 1.035)

    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_yticks([0.0, 0.5, 1.0])

    ax.set_aspect("equal", adjustable="box")


# ============================================================
# AXIS LABELS
# ============================================================
axes[0].set_ylabel("TSA", labelpad=8)

fig.supxlabel(
    "CAAL",
    y=0.105,
    fontsize=11,
)


# ============================================================
# SUBTLE DIAGONAL LABEL
# ============================================================
axes[-1].text(
    0.58,
    0.68,
    "TSA = CAAL",
    rotation=38,
    color="#777777",
    fontsize=8.5,
    alpha=0.75,
    transform=axes[-1].transAxes,
)


# ============================================================
# TITLE
# ============================================================
fig.suptitle(
    "TSA vs. CAAL Across Experiments",
    y=0.965,
    fontsize=15,
    fontweight="semibold",
    color="#111111",
)


# ============================================================
# HORIZONTAL LEGEND BAND
# ============================================================
all_handles = model_legend_handles() + size_legend_handles()

fig.legend(
    handles=all_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncol=11,
    frameon=False,
    columnspacing=0.85,
    handletextpad=0.28,
    borderaxespad=0.0,
    labelspacing=0.35,
)


# ============================================================
# LAYOUT
# ============================================================
fig.subplots_adjust(
    left=0.055,
    right=0.992,
    top=0.84,
    bottom=0.20,
    wspace=0.18,
)


# ============================================================
# SAVE + SHOW
# ============================================================
fig.savefig(
    OUTDIR / "appendix_scatter_tsa_caal_ttd.pdf",
    bbox_inches="tight",
    pad_inches=0.025,
)

fig.savefig(
    OUTDIR / "appendix_scatter_tsa_caal_ttd.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.025,
)

plt.show()

In [ ]:
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# CONFIG
# ============================================================
RESULTS_CSV = CSV_PATH
OUTDIR = Path("paper_assets/main_plots")
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ORDER = ["e1", "e2", "e3", "e5", "e7"]

EXPERIMENT_LABELS = {
    "e1": "E1",
    "e2": "E2",
    "e3": "E3",
    "e5": "E4",
    "e7": "E5",
}

MODEL_STYLES = OrderedDict([
    ("anthropic_claude-sonnet-4.6", {"label": "Claude Sonnet 4.6", "color": "#0072B2"}),
    ("qwen_qwen3.6-plus",           {"label": "Qwen 3.6+",         "color": "#D55E00"}),
    ("gemma4_26b",                  {"label": "Gemma 4 26B",       "color": "#009E73"}),
    ("gemma4_31b",                  {"label": "Gemma 4 31B",       "color": "#CC79A7"}),
    ("gpt-5.1",                     {"label": "GPT-5.1",           "color": "#E69F00"}),
    ("gemini-2.5-flash",            {"label": "Gemini 2.5 Flash",  "color": "#56B4E9"}),
    ("gemini-2.5-pro",              {"label": "Gemini 2.5 Pro",    "color": "#6A3D9A"}),
    ("ministral-3_14b",             {"label": "Ministral-3 14B",   "color": "#6B6B6B"}),
])

MODEL_ORDER = list(MODEL_STYLES.keys())

METRICS = OrderedDict([
    ("TSA",   {"raw_col": "TSA_mean",           "fmt": "{:.2f}", "mode": "direct"}),
    ("TTD",   {"raw_col": "TTD_mean",           "fmt": "{:.1f}", "mode": "reverse_within_exp"}),
    ("MCR",   {"raw_col": "MCR_mean",           "fmt": "{:.2f}", "mode": "direct"}),
    ("MTATO", {"raw_col": "MTATO_mean",         "fmt": "{:.2f}", "mode": "direct"}),
    ("CAAL",  {"raw_col": "CAAL_mean",          "fmt": "{:.2f}", "mode": "direct"}),
    ("CR",    {"raw_col": "CONFABULATION_mean", "fmt": "{:.2f}", "mode": "reverse_unit"}),
])


# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 9,
    "figure.dpi": 160,
    "savefig.dpi": 300,
})


# ============================================================
# LOAD
# ============================================================
df = pd.read_csv(RESULTS_CSV)
df = df[df["experiment_id"].isin(EXPERIMENT_ORDER)].copy()


# ============================================================
# HELPERS
# ============================================================
def normalize_series(series, mode):
    s = series.astype(float)

    if mode == "direct":
        return s.clip(0, 1)

    if mode == "reverse_unit":
        return (1 - s).clip(0, 1)

    if mode == "reverse_within_exp":
        valid = s.dropna()

        if len(valid) == 0:
            return pd.Series(np.nan, index=s.index)

        mn, mx = valid.min(), valid.max()

        if np.isclose(mx, mn):
            return pd.Series(0.5, index=s.index)

        return (1 - (s - mn) / (mx - mn)).clip(0, 1)

    raise ValueError(f"Unknown mode: {mode}")


def annotation_color(value):
    if pd.isna(value):
        return "#666666"

    return "white" if value >= 0.62 else "#222222"


# ============================================================
# BUILD PER-EXPERIMENT MATRICES
# ============================================================
exp_scores = OrderedDict()
exp_annots = OrderedDict()

for exp in EXPERIMENT_ORDER:
    exp_label = EXPERIMENT_LABELS[exp]

    sub = (
        df[df["experiment_id"] == exp]
        .set_index("model_name")
        .reindex(MODEL_ORDER)
    )

    score_mat = pd.DataFrame(index=MODEL_ORDER)
    annot_mat = pd.DataFrame(index=MODEL_ORDER)

    for metric_name, spec in METRICS.items():
        raw = sub[spec["raw_col"]]
        score = normalize_series(raw, spec["mode"])
        fmt = spec.get("fmt", "{:.2f}")

        score_mat[metric_name] = score
        annot_mat[metric_name] = [
            "—" if pd.isna(v) else fmt.format(v)
            for v in raw
        ]

    exp_scores[exp_label] = score_mat
    exp_annots[exp_label] = annot_mat

row_labels = [MODEL_STYLES[m]["label"] for m in MODEL_ORDER]


# ============================================================
# COLORMAP
# ============================================================
cmap = LinearSegmentedColormap.from_list(
    "elegant_score",
    ["#F7F7F7", "#DCEDE7", "#8EC5B6", "#3B8EA5", "#1D4E89"],
)
cmap.set_bad("#FFFFFF")


# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(18, 6))
fig.patch.set_facecolor("white")

gs = fig.add_gridspec(
    1,
    len(EXPERIMENT_ORDER) + 1,
    width_ratios=[6, 6, 6, 6, 6, 0.45],
    wspace=0.18,
)

axes = [
    fig.add_subplot(gs[0, i])
    for i in range(len(EXPERIMENT_ORDER))
]

cax = fig.add_subplot(gs[0, len(EXPERIMENT_ORDER)])


# ============================================================
# MAIN HEATMAPS
# ============================================================
for i, (ax_i, exp_label) in enumerate(zip(axes, exp_scores.keys())):
    show_y = i == 0

    sns.heatmap(
        exp_scores[exp_label],
        ax=ax_i,
        cmap=cmap,
        vmin=0,
        vmax=1,
        annot=exp_annots[exp_label].values,
        fmt="",
        linewidths=0.45,
        linecolor="#FFFFFF",
        cbar=i == len(axes) - 1,
        cbar_ax=cax if i == len(axes) - 1 else None,
        annot_kws={
            "fontsize": 8,
            "fontweight": "regular",
        },
    )

    for text, value in zip(ax_i.texts, exp_scores[exp_label].to_numpy().ravel()):
        text.set_color(annotation_color(value))

    ax_i.set_title(
        exp_label,
        fontweight="semibold",
        color="#222222",
        pad=10,
    )

    ax_i.set_xlabel("")
    ax_i.set_ylabel("")
    ax_i.tick_params(axis="both", length=0)

    ax_i.set_xticklabels(
        ax_i.get_xticklabels(),
        rotation=0,
        ha="center",
    )

    if show_y:
        ax_i.set_yticklabels(row_labels, rotation=0)

        for tick, model_id in zip(ax_i.get_yticklabels(), MODEL_ORDER):
            tick.set_color(MODEL_STYLES[model_id]["color"])
            tick.set_fontweight("semibold")
    else:
        ax_i.set_yticklabels([])
        ax_i.set_ylabel("")

    for spine in ax_i.spines.values():
        spine.set_visible(False)


# ============================================================
# COLORBAR
# ============================================================


cax.tick_params(labelsize=8, length=3, width=0.6)

for spine in cax.spines.values():
    spine.set_visible(False)


# ============================================================
# TITLE
# ============================================================
fig.suptitle(
    "Model Performance Across Experiments and Metrics",
    fontsize=16,
    fontweight="semibold",
    color="#111111",
    y=0.985,
)


# ============================================================
# LAYOUT
# ============================================================
fig.subplots_adjust(
    left=0.13,
    right=0.97,
    top=0.86,
    bottom=0.15,
)


# ============================================================
# SAVE + SHOW
# ============================================================
fig.savefig(
    OUTDIR / "Heatmap_final.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTDIR / "model_performance_heatmap_no_overall.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
from pathlib import Path
from collections import OrderedDict
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# CONFIG
# ============================================================
RESULTS_CSV = CSV_PATH
OUTDIR = Path("paper_assets/appendix_plots")
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ORDER = ["e1", "e2", "e3", "e5", "e7"]

EXPERIMENT_LABELS = {
    "e1": "E1",
    "e2": "E2",
    "e3": "E3",
    "e5": "E4",
    "e7": "E5",
}

MODEL_STYLES = OrderedDict([
    ("anthropic_claude-sonnet-4.6", {"label": "Claude Sonnet 4.6",    "color": "#0072B2", "marker": "o"}),
    ("qwen_qwen3.6-plus",           {"label": "Qwen 3.6+",     "color": "#D55E00", "marker": "s"}),
    ("gemma4_26b",                  {"label": "Gemma 4 26B",     "color": "#009E73", "marker": "^"}),
    ("gemma4_31b",                  {"label": "Gemma 4 31B",     "color": "#CC79A7", "marker": "D"}),
    ("gpt-5.1",                     {"label": "GPT-5.1",       "color": "#E69F00", "marker": "P"}),
    ("gemini-2.5-flash",            {"label": "Gemini 2.5 Flash",  "color": "#56B4E9", "marker": "X"}),
    ("gemini-2.5-pro",              {"label": "Gemini 2.5 Pro",    "color": "#6A3D9A", "marker": "v"}),
    ("ministral-3_14b",             {"label": "Ministral-3 14B", "color": "#6B6B6B", "marker": "h"}),
])

MODEL_ORDER = list(MODEL_STYLES.keys())


# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10,
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
})


# ============================================================
# LOAD
# ============================================================
df = pd.read_csv(RESULTS_CSV)
df = df[df["experiment_id"].isin(EXPERIMENT_ORDER)].copy()
df["paper_exp"] = df["experiment_id"].map(EXPERIMENT_LABELS)


# ============================================================
# HELPERS
# ============================================================
def style_axis(ax):
    ax.set_facecolor("#FAFAF8")

    ax.grid(
        True,
        axis="x",
        color="#D9D9D6",
        linewidth=0.65,
        alpha=0.55,
    )
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color("#777777")
    ax.spines["bottom"].set_color("#777777")
    ax.spines["left"].set_linewidth(0.75)
    ax.spines["bottom"].set_linewidth(0.75)

    ax.tick_params(
        axis="both",
        length=3,
        width=0.75,
        colors="#333333",
        pad=3,
    )


def parse_json_field(value):
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    return json.loads(value)


def e5_outcome_decomposition(row):
    counts = parse_json_field(row["ident_confusion_counts_json"])
    total = int(row["total_runs"])

    unknown = 0
    correct = 0

    for true_label, pred_dict in counts.items():
        for pred_label, c in pred_dict.items():
            if pred_label == "unknown_identification":
                unknown += c

        if true_label in pred_dict:
            correct += pred_dict[true_label]

    wrong = total - correct - unknown

    return {
        "correct": correct / total if total else np.nan,
        "wrong": wrong / total if total else np.nan,
        "unknown": unknown / total if total else np.nan,
        "correct_n": correct,
        "wrong_n": wrong,
        "unknown_n": unknown,
    }


def model_legend_handles():
    handles = []

    for model_id, spec in MODEL_STYLES.items():
        handles.append(
            Line2D(
                [0],
                [0],
                color="none",
                marker=spec["marker"],
                linestyle="None",
                markersize=7.2,
                markerfacecolor=spec["color"],
                markeredgecolor="white",
                markeredgewidth=0.9,
                label=spec["label"],
            )
        )

    return handles


# ============================================================
# BUILD PLOT DATA
# ============================================================
sub = (
    df[df["experiment_id"] == "e7"]
    .set_index("model_name")
    .reindex(MODEL_ORDER)
    .reset_index()
)

rows = []
for _, row in sub.iterrows():
    if pd.isna(row["model_name"]):
        continue

    dec = e5_outcome_decomposition(row)

    rows.append({
        "model_name": row["model_name"],
        **dec,
    })

plot_df = pd.DataFrame(rows)

plot_df["model_label"] = plot_df["model_name"].map(
    lambda m: MODEL_STYLES[m]["label"]
)

ys = np.arange(len(plot_df))


# ============================================================
# COLORS
# ============================================================
correct_color = "#5A9B58"
wrong_color = "#D47A2A"
unknown_color = "#D2D2CF"

bar_specs = [
    ("correct", correct_color, "Correct"),
    ("wrong", wrong_color, "Wrong committed"),
    ("unknown", unknown_color, "Unknown / no committed identification"),
]


# ============================================================
# FIGURE
# ============================================================
fig, ax = plt.subplots(figsize=(10.8, 5.4))
fig.patch.set_facecolor("white")

style_axis(ax)


# ============================================================
# STACKED BARS
# ============================================================
left = np.zeros(len(plot_df))

for key, color, label in bar_specs:
    vals = plot_df[key].fillna(0).values

    ax.barh(
        ys,
        vals,
        left=left,
        height=0.66,
        color=color,
        alpha=0.92,
        edgecolor="white",
        linewidth=0.9,
        label=label,
        zorder=2,
    )

    for i, value in enumerate(vals):
        if value >= 0.075:
            ax.text(
                left[i] + value / 2,
                ys[i],
                f"{value:.0%}",
                ha="center",
                va="center",
                fontsize=8.5,
                color="white" if key in {"correct", "wrong"} else "#333333",
                fontweight="semibold",
                zorder=5,
            )

    left += vals


# ============================================================
# MODEL MARKERS
# ============================================================
for i, row in plot_df.iterrows():
    spec = MODEL_STYLES[row["model_name"]]

    ax.scatter(
        -0.045,
        ys[i],
        s=92,
        color=spec["color"],
        marker=spec["marker"],
        edgecolors="white",
        linewidths=0.9,
        clip_on=False,
        zorder=4,
    )


# ============================================================
# AXES
# ============================================================
ax.set_yticks(ys)
ax.set_yticklabels(plot_df["model_label"])

for tick, model_id in zip(ax.get_yticklabels(), plot_df["model_name"]):
    tick.set_color(MODEL_STYLES[model_id]["color"])
    tick.set_fontweight("semibold")

ax.set_xlim(-0.085, 1.02)
ax.set_xticks([0.0, 0.25, 0.50, 0.75, 1.0])
ax.set_xticklabels(["0", "25", "50", "75", "100"])

ax.set_xlabel("Fraction of runs (%)", labelpad=8)

ax.invert_yaxis()


# ============================================================
# TITLE
# ============================================================
ax.set_title(
    "E5: Final Identification Outcome Decomposition",
    fontweight="semibold",
    color="#111111",
    pad=14,
)


# ============================================================
# REFERENCE LINES
# ============================================================
for x in [0.25, 0.50, 0.75, 1.0]:
    ax.axvline(
        x,
        color="#D9D9D6",
        linewidth=0.65,
        alpha=0.45,
        zorder=0,
    )


# ============================================================
# LEGEND
# ============================================================
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.13),
    ncol=3,
    frameon=False,
    columnspacing=1.2,
    handlelength=1.6,
    handletextpad=0.5,
)


# ============================================================
# LAYOUT
# ============================================================
fig.subplots_adjust(
    left=0.20,
    right=0.985,
    top=0.86,
    bottom=0.22,
)


# ============================================================
# SAVE + SHOW
# ============================================================
fig.savefig(
    OUTDIR / "e5_identification_FINALpdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTDIR / "e5_identification_outcome_decomposition_elegant.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# CONFIG
# ============================================================
RESULTS_CSV = CSV_PATH
OUTDIR = Path("paper_assets/appendix_plots")
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ORDER = ["e1", "e2", "e3", "e5", "e7"]

EXPERIMENT_LABELS = {
    "e1": "E1",
    "e2": "E2",
    "e3": "E3",
    "e5": "E4",
    "e7": "E5",
}

PAPER_MODEL_NAMES = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "qwen_qwen3.6-plus": "Qwen 3.6 Plus",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT 5.1",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "ministral-3_14b": "Ministral 3 14B",
}

PAPER_MODEL_ORDER = [
    "Ministral 3 14B",
    "Gemma4 26B",
    "Gemma4 31B",
    "Gemini 2.5 Flash",
    "Gemini 2.5 Pro",
    "Qwen 3.6 Plus",
    "GPT 5.1",
    "Claude Sonnet 4.6",
]


# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})


# ============================================================
# LOAD
# ============================================================
df = pd.read_csv(RESULTS_CSV)

plot_df = df.copy()
plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENT_ORDER)].copy()
plot_df["paper_exp"] = plot_df["experiment_id"].map(EXPERIMENT_LABELS)
plot_df["paper_model"] = plot_df["model_name"].map(PAPER_MODEL_NAMES)

plot_df["delta_complete"] = (
    plot_df["TSA-COMPLETE_mean"] - plot_df["TSA_mean"]
)


# ============================================================
# BUILD RAW MATRIX
# ============================================================
delta_df = (
    plot_df
    .pivot(index="paper_model", columns="paper_exp", values="delta_complete")
    .reindex(
        index=PAPER_MODEL_ORDER,
        columns=[EXPERIMENT_LABELS[e] for e in EXPERIMENT_ORDER],
    )
)

annot_df = delta_df.map(lambda x: f"{x:.2f}" if pd.notna(x) else "—")


# ============================================================
# NORMALIZE FOR COLOR ONLY
# ============================================================
delta_min = np.nanmin(delta_df.values)
delta_max = np.nanmax(delta_df.values)

if np.isclose(delta_max, delta_min):
    delta_norm = delta_df.copy() * 0 + 0.5
else:
    delta_norm = (delta_df - delta_min) / (delta_max - delta_min)

delta_norm = delta_norm.clip(0, 1)


# ============================================================
# COLORMAP
# ============================================================
cmap = LinearSegmentedColormap.from_list(
    "completion_gap",
    ["#F7F7F7", "#FBE5D6", "#F4A261", "#D55E00", "#8C2D04"],
)
cmap.set_bad("#FFFFFF")


def annotation_color(value):
    if pd.isna(value):
        return "#666666"

    return "white" if value >= 0.58 else "#222222"


# ============================================================
# FIGURE
# ============================================================
fig, ax = plt.subplots(figsize=(8.4, 5.1))
fig.patch.set_facecolor("white")

sns.heatmap(
    delta_norm,
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=1,
    annot=annot_df.values,
    fmt="",
    linewidths=0.55,
    linecolor="#FFFFFF",
    cbar=True,
    annot_kws={
        "fontsize": 8.8,
        "fontweight": "semibold",
    },
)

for text, value in zip(ax.texts, delta_norm.to_numpy().ravel()):
    text.set_color(annotation_color(value))


# ============================================================
# AXIS STYLE
# ============================================================
ax.set_title(
    r"Completion Gap: $\mathrm{TSA}_{\mathrm{complete}} - \mathrm{TSA}$",
    pad=14,
    fontweight="semibold",
    color="#111111",
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xticklabels(
    delta_norm.columns,
    rotation=0,
    ha="center",
)

ax.set_yticklabels(
    delta_norm.index,
    rotation=0,
)

ax.tick_params(axis="both", length=0)

for spine in ax.spines.values():
    spine.set_visible(False)


# ============================================================
# COLORBAR STYLE, NORMALIZED SCALE
# ============================================================
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=8.5, length=3, width=0.6)
cbar.outline.set_visible(False)
cbar.ax.yaxis.label.set_size(9)

cbar.set_ticks([0.0, 0.25, 0.50, 0.75, 1.0])
cbar.set_ticklabels(["0.00", "0.25", "0.50", "0.75", "1.00"])



# ============================================================
# LAYOUT
# ============================================================
fig.subplots_adjust(
    left=0.24,
    right=0.96,
    top=0.87,
    bottom=0.15,
)


# ============================================================
# SAVE + SHOW
# ============================================================
fig.savefig(
    OUTDIR / "Tsa_Complete-Tsa.pdf",
    bbox_inches="tight",
    pad_inches=0.03,
)

fig.savefig(
    OUTDIR / "appendix_completion_gap_heatmap_normalized.png",
    dpi=500,
    bbox_inches="tight",
    pad_inches=0.03,
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# CONFIG
# ============================================================

EXPERIMENTS = ["e1", "e2", "e3", "e5"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e2": "No mirror trap",
    "e3": "Wrong prior",
    "e5": "Distractors",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
}

MODEL_MARKERS = {
    "Claude Sonnet 4.6": "o",
    "Gemini 2.5 Flash": "s",
    "Gemini 2.5 Pro": "^",
    "Gemma4 26B": "D",
    "Gemma4 31B": "P",
    "GPT-5.1": "X",
    "Ministral 3 14B": "v",
    "Qwen 3.6 Plus": "*",
}


# ============================================================
# CLEAN EXISTING DF
# ============================================================

plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = (
    plot_df["model_name"]
    .map(MODEL_LABELS)
    .fillna(plot_df["model_name"])
)

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

required = {"experiment_id", "model_label", "TSA_mean", "TTD_mean"}
missing = required - set(plot_df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()

model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

cmap = plt.get_cmap("tab10")
MODEL_COLORS = {
    model: cmap(i % 10)
    for i, model in enumerate(model_order)
}


# ============================================================
# PLOT: TTD vs TSA
# ============================================================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "figure.titlesize": 15,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.2, 8.8),
    sharex=True,
    sharey=True,
)

axes = axes.ravel()

x_min = max(0, plot_df["TTD_mean"].min() - 5)
x_max = plot_df["TTD_mean"].max() + 5

for ax, exp in zip(axes, EXPERIMENTS):
    sub = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    ax.axhline(
        1 / 3,
        linestyle="--",
        linewidth=1.1,
        color="0.25",
        alpha=0.65,
        zorder=1,
    )

    median_ttd = sub["TTD_mean"].median()
    ax.axvline(
        median_ttd,
        linestyle=":",
        linewidth=1.0,
        color="0.25",
        alpha=0.45,
        zorder=1,
    )

    for model in model_order:
        row = sub.loc[model]

        ax.scatter(
            row["TTD_mean"],
            row["TSA_mean"],
            s=95,
            marker=MODEL_MARKERS.get(model, "o"),
            facecolor=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.7,
            alpha=0.88,
            zorder=3,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=8)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-0.03, 1.03)
    ax.grid(alpha=0.18, linewidth=0.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.text(
    0.48,
    0.075,
    "Mean Time-to-Decision",
    ha="center",
    fontsize=12,
)

fig.text(
    0.045,
    0.53,
    "Target Selection Accuracy",
    va="center",
    rotation="vertical",
    fontsize=12,
)

fig.suptitle(
    "Impulsive vs effective decisions",
    y=0.965,
    fontsize=15,
)

model_handles = [
    Line2D(
        [0],
        [0],
        marker=MODEL_MARKERS.get(model, "o"),
        color="none",
        markerfacecolor=MODEL_COLORS[model],
        markeredgecolor="black",
        markeredgewidth=0.7,
        markersize=7,
        label=model,
    )
    for model in model_order
]

reference_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle="--",
        linewidth=1.1,
        label="Random target selection",
    ),
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle=":",
        linewidth=1.0,
        label="Panel median TTD",
    ),
]

fig.legend(
    handles=model_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.78),
    frameon=False,
    title="Model",
    title_fontsize=10,
)

fig.legend(
    handles=reference_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.34),
    frameon=False,
    title="Reference",
    title_fontsize=10,
)

fig.text(
    0.48,
    0.025,
    "Low TTD with low TSA suggests impulsive guessing; high TTD with low TSA suggests deliberation without effective self-recognition.",
    ha="center",
    fontsize=9,
)

fig.subplots_adjust(
    left=0.09,
    right=0.79,
    top=0.90,
    bottom=0.14,
    wspace=0.08,
    hspace=0.20,
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# CONFIG
# ============================================================

EXPERIMENTS = ["e1", "e2", "e3", "e5"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e2": "No mirror trap",
    "e3": "Wrong prior",
    "e5": "Distractors",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
}

MODEL_MARKERS = {
    "Claude Sonnet 4.6": "o",
    "Gemini 2.5 Flash": "s",
    "Gemini 2.5 Pro": "^",
    "Gemma4 26B": "D",
    "Gemma4 31B": "P",
    "GPT-5.1": "X",
    "Ministral 3 14B": "v",
    "Qwen 3.6 Plus": "*",
}


# ============================================================
# CLEAN EXISTING DF
# ============================================================

plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = (
    plot_df["model_name"]
    .map(MODEL_LABELS)
    .fillna(plot_df["model_name"])
)

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

required = {
    "experiment_id",
    "model_label",
    "TSA_mean",
    "CONFABULATION_mean",
    "TTD_mean",
}
missing = required - set(plot_df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()


# ============================================================
# ORDER, COLORS, SIZE
# ============================================================

model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

cmap = plt.get_cmap("tab10")
MODEL_COLORS = {
    model: cmap(i % 10)
    for i, model in enumerate(model_order)
}

ttd_min = plot_df["TTD_mean"].min()
ttd_max = plot_df["TTD_mean"].max()

def scale_ttd_to_size(ttd):
    if np.isclose(ttd_max, ttd_min):
        return 120
    return 60 + 240 * ((ttd - ttd_min) / (ttd_max - ttd_min))

plot_df["bubble_size"] = plot_df["TTD_mean"].apply(scale_ttd_to_size)


# ============================================================
# PLOT
# ============================================================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "figure.titlesize": 15,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.2, 8.8),
    sharex=True,
    sharey=True,
)

axes = axes.ravel()

for ax, exp in zip(axes, EXPERIMENTS):
    sub = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    # Random target selection baseline
    ax.axhline(
        1 / 3,
        linestyle="--",
        linewidth=1.1,
        color="0.25",
        alpha=0.65,
        zorder=1,
    )

    # High-confabulation region
    ax.axvspan(
        0.5,
        1.0,
        color="0.7",
        alpha=0.08,
        zorder=0,
    )

    if exp == "e2":
        ax.text(
            0.75,
            0.08,
            "unsupported\nself-assertion",
            ha="center",
            va="bottom",
            fontsize=9,
            color="0.25",
        )

    for model in model_order:
        row = sub.loc[model]

        ax.scatter(
            row["CONFABULATION_mean"],
            row["TSA_mean"],
            s=row["bubble_size"],
            marker=MODEL_MARKERS.get(model, "o"),
            facecolor=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.75,
            alpha=0.88,
            zorder=3,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=8)
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.grid(alpha=0.18, linewidth=0.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# Shared axis labels
fig.text(
    0.48,
    0.075,
    "Confabulation Rate",
    ha="center",
    fontsize=12,
)

fig.text(
    0.045,
    0.53,
    "Target Selection Accuracy",
    va="center",
    rotation="vertical",
    fontsize=12,
)

fig.suptitle(
    "Confabulation, task success, and decision time",
    y=0.965,
    fontsize=15,
)


# ============================================================
# LEGENDS
# ============================================================

model_handles = [
    Line2D(
        [0],
        [0],
        marker=MODEL_MARKERS.get(model, "o"),
        color="none",
        markerfacecolor=MODEL_COLORS[model],
        markeredgecolor="black",
        markeredgewidth=0.75,
        markersize=7,
        label=model,
    )
    for model in model_order
]

reference_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle="--",
        linewidth=1.1,
        label="Random target selection",
    ),
]

example_ttds = np.linspace(ttd_min, ttd_max, 3)
size_handles = [
    plt.scatter(
        [],
        [],
        s=scale_ttd_to_size(t),
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        label=f"{t:.0f} steps",
    )
    for t in example_ttds
]

fig.legend(
    handles=model_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.78),
    frameon=False,
    title="Model",
    title_fontsize=10,
)

fig.legend(
    handles=reference_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.36),
    frameon=False,
    title="Reference",
    title_fontsize=10,
)

fig.legend(
    handles=size_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.22),
    frameon=False,
    title="Mean TTD",
    title_fontsize=10,
)

fig.text(
    0.48,
    0.025,
    "Bubble size encodes mean time-to-decision. In the no-mirror trap, confabulation indicates self-color assertion without visual evidence.",
    ha="center",
    fontsize=9,
)

fig.subplots_adjust(
    left=0.09,
    right=0.79,
    top=0.90,
    bottom=0.14,
    wspace=0.08,
    hspace=0.20,
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# CONFIG
# ============================================================

EXPERIMENTS = ["e1", "e3", "e5", "e7"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e3": "Wrong prior",
    "e5": "Distractors",
    "e7": "Cluttered exploration",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
}

MODEL_MARKERS = {
    "Claude Sonnet 4.6": "o",
    "Gemini 2.5 Flash": "s",
    "Gemini 2.5 Pro": "^",
    "Gemma4 26B": "D",
    "Gemma4 31B": "P",
    "GPT-5.1": "X",
    "Ministral 3 14B": "v",
    "Qwen 3.6 Plus": "*",
}


# ============================================================
# CLEAN EXISTING DF
# ============================================================

plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = (
    plot_df["model_name"]
    .map(MODEL_LABELS)
    .fillna(plot_df["model_name"])
)

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

required = {
    "experiment_id",
    "model_label",
    "TSA_mean",
    "MGD_mean",
    "TTD_mean",
}

missing = required - set(plot_df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()


# ============================================================
# ORDER, COLORS, SIZE
# ============================================================

model_order = (
    plot_df[plot_df["experiment_id"] == "e1"]
    .sort_values("TSA_mean", ascending=False)["model_label"]
    .tolist()
)

# Fallback in case e1 is missing
if len(model_order) == 0:
    model_order = (
        plot_df.groupby("model_label")["TSA_mean"]
        .mean()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

cmap = plt.get_cmap("tab10")
MODEL_COLORS = {
    model: cmap(i % 10)
    for i, model in enumerate(model_order)
}

ttd_min = plot_df["TTD_mean"].min()
ttd_max = plot_df["TTD_mean"].max()

def scale_ttd_to_size(ttd):
    if np.isclose(ttd_max, ttd_min):
        return 120
    return 60 + 240 * ((ttd - ttd_min) / (ttd_max - ttd_min))

plot_df["bubble_size"] = plot_df["TTD_mean"].apply(scale_ttd_to_size)


# ============================================================
# PLOT
# ============================================================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "figure.titlesize": 15,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.2, 8.8),
    sharex=True,
    sharey=True,
)

axes = axes.ravel()

x_min = max(0, plot_df["MGD_mean"].min() - 1)
x_max = plot_df["MGD_mean"].max() + 1

for ax, exp in zip(axes, EXPERIMENTS):
    sub = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    # Random target selection baseline
    ax.axhline(
        1 / 3,
        linestyle="--",
        linewidth=1.1,
        color="0.25",
        alpha=0.65,
        zorder=1,
    )

    # Median mirror-gaze duration in this panel
    panel_median_mgd = sub["MGD_mean"].median()
    ax.axvline(
        panel_median_mgd,
        linestyle=":",
        linewidth=1.0,
        color="0.25",
        alpha=0.45,
        zorder=1,
    )

    for model in model_order:
        row = sub.loc[model]

        ax.scatter(
            row["MGD_mean"],
            row["TSA_mean"],
            s=row["bubble_size"],
            marker=MODEL_MARKERS.get(model, "o"),
            facecolor=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.75,
            alpha=0.88,
            zorder=3,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=8)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-0.03, 1.03)
    ax.grid(alpha=0.18, linewidth=0.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# Shared axis labels
fig.text(
    0.48,
    0.075,
    "Mirror Gaze Duration",
    ha="center",
    fontsize=12,
)

fig.text(
    0.045,
    0.53,
    "Target Selection Accuracy",
    va="center",
    rotation="vertical",
    fontsize=12,
)

fig.suptitle(
    "Mirror inspection, task success, and decision time",
    y=0.965,
    fontsize=15,
)


# ============================================================
# LEGENDS
# ============================================================

model_handles = [
    Line2D(
        [0],
        [0],
        marker=MODEL_MARKERS.get(model, "o"),
        color="none",
        markerfacecolor=MODEL_COLORS[model],
        markeredgecolor="black",
        markeredgewidth=0.75,
        markersize=7,
        label=model,
    )
    for model in model_order
]

reference_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle="--",
        linewidth=1.1,
        label="Random target selection",
    ),
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle=":",
        linewidth=1.0,
        label="Panel median MGD",
    ),
]

example_ttds = np.linspace(ttd_min, ttd_max, 3)
size_handles = [
    plt.scatter(
        [],
        [],
        s=scale_ttd_to_size(t),
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        label=f"{t:.0f} steps",
    )
    for t in example_ttds
]

fig.legend(
    handles=model_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.78),
    frameon=False,
    title="Model",
    title_fontsize=10,
)

fig.legend(
    handles=reference_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.36),
    frameon=False,
    title="Reference",
    title_fontsize=10,
)

fig.legend(
    handles=size_handles,
    loc="upper left",
    bbox_to_anchor=(0.825, 0.20),
    frameon=False,
    title="Mean TTD",
    title_fontsize=10,
)

fig.text(
    0.48,
    0.025,
    "Bubble size encodes mean time-to-decision. Higher mirror gaze does not necessarily imply better self-recognition.",
    ha="center",
    fontsize=9,
)

fig.subplots_adjust(
    left=0.09,
    right=0.79,
    top=0.90,
    bottom=0.14,
    wspace=0.08,
    hspace=0.20,
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

EXPERIMENTS = ["e1", "e2", "e3", "e5", "e7"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e2": "No mirror trap",
    "e3": "Wrong prior",
    "e5": "Distractors",
    "e7": "Cluttered",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
}


# ============================================================
# CLEAN EXISTING DF
# ============================================================

plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = (
    plot_df["model_name"]
    .map(MODEL_LABELS)
    .fillna(plot_df["model_name"])
)

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

required = {"experiment_id", "model_label", "RCS_mean", "TSA_mean", "TTD_mean"}
missing = required - set(plot_df.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()


# ============================================================
# DERIVED METRIC
# ============================================================

plot_df["stable_wrongness"] = plot_df["RCS_mean"] * (1 - plot_df["TSA_mean"])

# Optional: decision-time weighted variant
plot_df["stable_wrongness_ttd"] = plot_df["stable_wrongness"] * plot_df["TTD_mean"]


# ============================================================
# MODEL ORDER
# ============================================================

model_order = (
    plot_df.groupby("model_label")["stable_wrongness"]
    .mean()
    .sort_values(ascending=True)
    .index
    .tolist()
)

plot_df["model_label"] = pd.Categorical(
    plot_df["model_label"],
    categories=model_order,
    ordered=True,
)

plot_df["experiment_id"] = pd.Categorical(
    plot_df["experiment_id"],
    categories=EXPERIMENTS,
    ordered=True,
)


# ============================================================
# PLOT
# ============================================================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "figure.titlesize": 14,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

fig, axes = plt.subplots(
    1,
    len(EXPERIMENTS),
    figsize=(15.5, 5.0),
    sharey=True,
    sharex=True,
)

y = np.arange(len(model_order))

for ax, exp in zip(axes, EXPERIMENTS):
    sub = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
    )

    x = sub["stable_wrongness"].values

    ax.hlines(
        y,
        xmin=0,
        xmax=x,
        color="0.55",
        linewidth=1.4,
        alpha=0.75,
        zorder=1,
    )

    ax.scatter(
        x,
        y,
        s=70,
        color="black",
        edgecolor="black",
        zorder=3,
    )

    ax.set_title(EXPERIMENT_LABELS[exp])
    ax.set_xlim(0, 1.0)
    ax.grid(axis="x", alpha=0.22)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_yticks(y)
axes[0].set_yticklabels(model_order)

for ax in axes[1:]:
    ax.tick_params(axis="y", length=0)

fig.supxlabel("Stable Wrongness Score = RCS × (1 - TSA)", y=0.04)
fig.suptitle(
    "Stable wrongness across task variants",
    y=1.02,
)

fig.text(
    0.5,
    -0.02,
    "Higher values indicate models that maintain a coherent self-description while failing the task.",
    ha="center",
    fontsize=9,
)

fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# CONFIG
# ============================================================

EXPERIMENTS = ["e1", "e3", "e5"]

EXPERIMENT_LABELS = {
    "e1": "Mirror",
    "e3": "Wrong prior",
    "e5": "Distractors",
}

MODEL_LABELS = {
    "anthropic_claude-sonnet-4.6": "Claude Sonnet 4.6",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemma4_26b": "Gemma4 26B",
    "gemma4_31b": "Gemma4 31B",
    "gpt-5.1": "GPT-5.1",
    "ministral_3_14b": "Ministral 3 14B",
    "qwen3_6_plus": "Qwen 3.6 Plus",
    "qwen_qwen3.6-plus": "Qwen 3.6 Plus",
}

MODEL_MARKERS = {
    "Claude Sonnet 4.6": "o",
    "Gemini 2.5 Flash": "s",
    "Gemini 2.5 Pro": "^",
    "Gemma4 26B": "D",
    "Gemma4 31B": "P",
    "GPT-5.1": "X",
    "Ministral 3 14B": "v",
    "Qwen 3.6 Plus": "*",
}


# ============================================================
# CLEAN EXISTING DF
# ============================================================

plot_df = df.copy()
plot_df.columns = plot_df.columns.astype(str).str.strip()

plot_df["model_label"] = (
    plot_df["model_name"]
    .map(MODEL_LABELS)
    .fillna(plot_df["model_name"])
)

for col in plot_df.columns:
    if col not in ["experiment_id", "model_name", "model_label"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

required = {
    "experiment_id",
    "model_label",
    "CONFABULATION_mean",
    "SELF_CORRECTION_mean",
    "SELF_CORRECTION_applicable_runs",
    "SELF_CORRECTION_applicable_rate",
}

missing = required - set(plot_df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns are:\n{list(plot_df.columns)}"
    )

plot_df = plot_df[plot_df["experiment_id"].isin(EXPERIMENTS)].copy()

# Keep only rows where self-correction was applicable.
# Important: NaN means "not applicable", not zero.
plot_df = plot_df[plot_df["SELF_CORRECTION_applicable_runs"] > 0].copy()


# ============================================================
# ORDER, COLORS, SIZE
# ============================================================

model_order = (
    plot_df.groupby("model_label")["SELF_CORRECTION_applicable_runs"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

cmap = plt.get_cmap("tab10")
MODEL_COLORS = {
    model: cmap(i % 10)
    for i, model in enumerate(model_order)
}

runs_min = plot_df["SELF_CORRECTION_applicable_runs"].min()
runs_max = plot_df["SELF_CORRECTION_applicable_runs"].max()

def scale_runs_to_size(n):
    if np.isclose(runs_max, runs_min):
        return 130
    return 70 + 280 * ((n - runs_min) / (runs_max - runs_min))

plot_df["bubble_size"] = plot_df["SELF_CORRECTION_applicable_runs"].apply(scale_runs_to_size)


# ============================================================
# PLOT
# ============================================================

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "figure.titlesize": 15,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

fig, axes = plt.subplots(
    1,
    len(EXPERIMENTS),
    figsize=(13.8, 4.8),
    sharex=True,
    sharey=True,
)

if len(EXPERIMENTS) == 1:
    axes = [axes]

for ax, exp in zip(axes, EXPERIMENTS):
    sub = (
        plot_df[plot_df["experiment_id"] == exp]
        .set_index("model_label")
        .reindex(model_order)
        .dropna(subset=["CONFABULATION_mean", "SELF_CORRECTION_mean"])
    )

    # Reference lines
    ax.axhline(
        0.5,
        linestyle=":",
        linewidth=1.0,
        color="0.25",
        alpha=0.45,
        zorder=1,
    )

    ax.axvline(
        0.5,
        linestyle=":",
        linewidth=1.0,
        color="0.25",
        alpha=0.45,
        zorder=1,
    )

    # High-risk quadrant: confabulates a lot, does not recover
    ax.axvspan(
        0.5,
        1.0,
        ymin=0.0,
        ymax=0.5,
        color="0.7",
        alpha=0.12,
        zorder=0,
    )

    ax.text(
        0.75,
        0.08,
        "anchored\nwrong belief",
        ha="center",
        va="bottom",
        fontsize=9,
        color="0.25",
    )

    for model in sub.index:
        row = sub.loc[model]

        ax.scatter(
            row["CONFABULATION_mean"],
            row["SELF_CORRECTION_mean"],
            s=row["bubble_size"],
            marker=MODEL_MARKERS.get(model, "o"),
            facecolor=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.75,
            alpha=0.88,
            zorder=3,
        )

    ax.set_title(EXPERIMENT_LABELS[exp], pad=8)
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.grid(alpha=0.18, linewidth=0.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# Shared labels
fig.text(
    0.48,
    0.08,
    "Confabulation Rate",
    ha="center",
    fontsize=12,
)

fig.text(
    0.045,
    0.53,
    "Self-Correction Rate",
    va="center",
    rotation="vertical",
    fontsize=12,
)

fig.suptitle(
    "Can models revise wrong early self-beliefs after mirror evidence?",
    y=0.98,
    fontsize=15,
)


# ============================================================
# LEGENDS
# ============================================================

model_handles = [
    Line2D(
        [0],
        [0],
        marker=MODEL_MARKERS.get(model, "o"),
        color="none",
        markerfacecolor=MODEL_COLORS[model],
        markeredgecolor="black",
        markeredgewidth=0.75,
        markersize=7,
        label=model,
    )
    for model in model_order
]

reference_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linestyle=":",
        linewidth=1.0,
        label="0.5 reference",
    ),
]

example_runs = np.linspace(runs_min, runs_max, 3)
example_runs = np.unique(np.round(example_runs).astype(int))

size_handles = [
    plt.scatter(
        [],
        [],
        s=scale_runs_to_size(n),
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        label=f"{n} runs",
    )
    for n in example_runs
]

fig.legend(
    handles=model_handles,
    loc="upper left",
    bbox_to_anchor=(0.83, 0.80),
    frameon=False,
    title="Model",
    title_fontsize=10,
)

fig.legend(
    handles=reference_handles,
    loc="upper left",
    bbox_to_anchor=(0.83, 0.38),
    frameon=False,
    title="Reference",
    title_fontsize=10,
)

fig.legend(
    handles=size_handles,
    loc="upper left",
    bbox_to_anchor=(0.83, 0.24),
    frameon=False,
    title="Applicable runs",
    title_fontsize=10,
)

fig.text(
    0.48,
    0.025,
    "Self-correction is computed only on applicable runs: wrong non-unknown self-color guessed before first mirror evidence.",
    ha="center",
    fontsize=9,
)

fig.subplots_adjust(
    left=0.09,
    right=0.80,
    top=0.86,
    bottom=0.18,
    wspace=0.10,
)

plt.show()